# JX — jQuery for JAX
> **Fluent, lazy-evaluation interface for tensor computation**

Author: J. Roberto Jiménez · `tijuanapaint@gmail.com`

---

## Philosophy

JX borrows the **fluent chaining** style of jQuery and applies it to JAX tensor pipelines.
You build a computation graph lazily — nothing executes until you explicitly ask for it.

```
$(x).normalize().relu().sum().jit()
```

The same chain works as eager code during debugging and as a compiled, batched, differentiable
function in production — just by changing the execution call at the end.

---

### Table of Contents
1. Installation & Setup
2. Core Concepts — Lazy Pipelines
3. Execution Strategies — `value_of`, `jit`, `vmap`, `pmap`
4. Autograd — `grad`, `value_and_grad`, `hessian`, `jacrev`
5. Tensor Operations
6. Binary Operations & Operator Overloading
7. Neural Network Layers
8. Loss Functions
9. Optimizers & Training Loops
10. Debugging & Graph Inspection
11. End-to-End Example — Linear Regression
12. End-to-End Example — MLP on Synthetic Data

## 1. Installation & Setup

In [ ]:
# Install dependencies (run once)
# !pip install jax jaxlib optax

In [ ]:
# ── paste jx_core.py content here, or import it if on your PYTHONPATH ──
# from jx.core import jx, $, JXTensor, Linear, Sequential, adam, sgd, adamw
# from jx.losses import mse, cross_entropy, binary_cross_entropy, l1_loss

# For this notebook we inline the full library:
import jax
import jax.numpy as jnp
from jax import jit, grad, vmap, pmap
from functools import reduce
from typing import Any, Callable, List, Dict, Optional, Union, Tuple
import hashlib
import optax

print(f"JAX version  : {jax.__version__}")
print(f"Devices      : {jax.devices()}")

In [ ]:
# ============================================================
# JX Library — full source (fixed build)
# ============================================================

class JXTensor:
    """
    Lazy tensor representation.
    Builds a computation graph without executing until you call
    .value_of(), .jit(), or another execution strategy.
    """

    def __init__(self, data=None, transforms=None, name=None,
                 batch_dim=None, device_count=None, requires_grad=False):
        self.data = data
        self.transforms = transforms or []
        self.name = name or f"jx_{id(self)}"
        self.batch_dim = batch_dim
        self.device_count = device_count
        self.requires_grad = requires_grad
        self._graph_hash = None

    def _add(self, fn, fn_name=None):
        label = fn_name or getattr(fn, '__name__', repr(fn))
        return JXTensor(
            data=self.data,
            transforms=self.transforms + [(fn, label)],
            batch_dim=self.batch_dim,
            device_count=self.device_count,
            requires_grad=self.requires_grad,
        )

    def _compose(self, x):
        return reduce(lambda acc, t: t[0](acc), self.transforms, x)

    def _clone(self, **overrides):
        return JXTensor(
            data=overrides.get('data', self.data),
            transforms=overrides.get('transforms', self.transforms),
            name=overrides.get('name', self.name),
            batch_dim=overrides.get('batch_dim', self.batch_dim),
            device_count=overrides.get('device_count', self.device_count),
            requires_grad=overrides.get('requires_grad', self.requires_grad),
        )

    @property
    def graph_hash(self):
        if self._graph_hash is None:
            labels = str([label for _, label in self.transforms])
            self._graph_hash = hashlib.md5(labels.encode()).hexdigest()[:8]
        return self._graph_hash

    # ── Execution ────────────────────────────────────────────
    def value_of(self):
        if self.data is None:
            raise ValueError("No input data — provide data via jx(data) first.")
        return self._compose(self.data)

    def jit(self, **kwargs):
        compiled = jit(self._compose, **kwargs)
        if self.data is not None:
            return compiled(self.data)
        return JXTensor(data=None,
                        transforms=[(compiled, f"jit_{self.graph_hash}")],
                        batch_dim=self.batch_dim)

    def vmap(self, in_axes=0, out_axes=0):
        pipeline = self._compose
        return self._clone(transforms=self.transforms + [
            (lambda x: vmap(pipeline, in_axes=in_axes, out_axes=out_axes)(x), "vmap")
        ], batch_dim=in_axes)

    def pmap(self, devices=None):
        n = devices or jax.device_count()
        pipeline = self._compose
        return self._clone(transforms=self.transforms + [
            (lambda x: pmap(pipeline)(x), "pmap")
        ], batch_dim=0, device_count=n)

    # ── Autograd ─────────────────────────────────────────────
    def grad(self, argnums=0, has_aux=False):
        grad_fn = grad(self._compose, argnums=argnums, has_aux=has_aux)
        return self._clone(transforms=[(grad_fn, f"grad_{self.graph_hash}")])

    def value_and_grad(self, argnums=0):
        vg_fn = jax.value_and_grad(self._compose, argnums=argnums)
        val_t  = self._clone(transforms=[(lambda x: vg_fn(x)[0], f"val_{self.graph_hash}")])
        grad_t = self._clone(transforms=[(lambda x: vg_fn(x)[1], f"grad_{self.graph_hash}")])
        return val_t, grad_t

    def jacrev(self):
        from jax import jacrev as _jr
        p = self._compose
        return self._add(lambda x: _jr(p)(x), "jacrev")

    def jacfwd(self):
        from jax import jacfwd as _jf
        p = self._compose
        return self._add(lambda x: _jf(p)(x), "jacfwd")

    def hessian(self):
        p = self._compose
        return self._add(lambda x: jax.hessian(p)(x), "hessian")

    # ── Shape ────────────────────────────────────────────────
    def __getitem__(self, idx):  return self._add(lambda x: x[idx], f"slice")
    def reshape(self, *shape):   return self._add(lambda x: x.reshape(*shape), f"reshape")
    def transpose(self, *axes):  return self._add(lambda x: x.transpose(*axes), "transpose")
    def squeeze(self, axis=None):return self._add(lambda x: x.squeeze(axis=axis), "squeeze")
    def unsqueeze(self, axis):   return self._add(lambda x: jnp.expand_dims(x, axis), "unsqueeze")

    # ── Activations ──────────────────────────────────────────
    def normalize(self, eps=1e-8):
        def _n(x): return (x - x.mean()) / jnp.sqrt(x.var() + eps)
        return self._add(_n, "normalize")
    def relu(self):               return self._add(jax.nn.relu, "relu")
    def sigmoid(self):            return self._add(jax.nn.sigmoid, "sigmoid")
    def log_sigmoid(self):        return self._add(jax.nn.log_sigmoid, "log_sigmoid")
    def tanh(self):               return self._add(jnp.tanh, "tanh")
    def gelu(self):               return self._add(jax.nn.gelu, "gelu")
    def softmax(self, axis=-1):   return self._add(lambda x: jax.nn.softmax(x, axis=axis), f"softmax")
    def log_softmax(self, axis=-1):return self._add(lambda x: jax.nn.log_softmax(x, axis=axis), "log_softmax")
    def abs(self):   return self._add(jnp.abs,  "abs")
    def sqrt(self):  return self._add(jnp.sqrt, "sqrt")
    def exp(self):   return self._add(jnp.exp,  "exp")
    def log(self):   return self._add(jnp.log,  "log")
    def sin(self):   return self._add(jnp.sin,  "sin")
    def cos(self):   return self._add(jnp.cos,  "cos")

    # ── Reductions ───────────────────────────────────────────
    def sum(self, axis=None, keepdims=False):
        return self._add(lambda x: jnp.sum(x, axis=axis, keepdims=keepdims), "sum")
    def mean(self, axis=None, keepdims=False):
        return self._add(lambda x: jnp.mean(x, axis=axis, keepdims=keepdims), "mean")
    def var(self, axis=None, keepdims=False):
        return self._add(lambda x: jnp.var(x, axis=axis, keepdims=keepdims), "var")
    def std(self, axis=None, keepdims=False):
        return self._add(lambda x: jnp.std(x, axis=axis, keepdims=keepdims), "std")
    def max(self, axis=None, keepdims=False):
        return self._add(lambda x: jnp.max(x, axis=axis, keepdims=keepdims), "max")
    def min(self, axis=None, keepdims=False):
        return self._add(lambda x: jnp.min(x, axis=axis, keepdims=keepdims), "min")
    def argmax(self, axis=None): return self._add(lambda x: jnp.argmax(x, axis=axis), "argmax")
    def argmin(self, axis=None): return self._add(lambda x: jnp.argmin(x, axis=axis), "argmin")

    # ── Binary ops ───────────────────────────────────────────
    @staticmethod
    def _resolve(other, x):
        if isinstance(other, JXTensor):
            return other.value_of() if other.data is not None else other._compose(x)
        return other

    def __add__(self, o):  return self._add(lambda x: x + JXTensor._resolve(o, x), "add")
    def __radd__(self, o): return self._add(lambda x: JXTensor._resolve(o, x) + x, "radd")
    def __sub__(self, o):  return self._add(lambda x: x - JXTensor._resolve(o, x), "sub")
    def __rsub__(self, o): return self._add(lambda x: JXTensor._resolve(o, x) - x, "rsub")
    def __mul__(self, o):  return self._add(lambda x: x * JXTensor._resolve(o, x), "mul")
    def __rmul__(self, o): return self._add(lambda x: JXTensor._resolve(o, x) * x, "rmul")
    def __truediv__(self, o): return self._add(lambda x: x / JXTensor._resolve(o, x), "div")
    def __pow__(self, o):  return self._add(lambda x: x ** JXTensor._resolve(o, x), "pow")
    def __matmul__(self, o):  return self._add(lambda x: x @ JXTensor._resolve(o, x), "matmul")
    def __neg__(self):     return self._add(lambda x: -x, "neg")

    def dot(self, other):    return self @ other
    def matmul(self, other): return self @ other

    def einsum(self, subscripts, *operands):
        def _e(x):
            ops = [o.value_of() if isinstance(o, JXTensor) and o.data is not None
                   else o._compose(x) if isinstance(o, JXTensor) else o
                   for o in operands]
            return jnp.einsum(subscripts, x, *ops)
        return self._add(_e, f"einsum_{subscripts}")

    # ── Debug ────────────────────────────────────────────────
    def debug_print(self, msg=""):
        def _d(x):
            print(f"[jx] {msg}  shape={x.shape}  dtype={x.dtype}  mean={float(jnp.mean(x)):.4f}")
            return x
        return self._add(_d, f"debug")

    def inspect(self):
        return {
            'name': self.name,
            'transforms': [l for _, l in self.transforms],
            'depth': len(self.transforms),
            'batch_dim': self.batch_dim,
            'devices': self.device_count,
            'requires_grad': self.requires_grad,
            'graph_hash': self.graph_hash,
            'has_data': self.data is not None,
        }

    def __repr__(self):
        s = '✓' if self.data is not None else '⋯'
        labels = ', '.join(l for _, l in self.transforms[:4])
        tail = '...' if len(self.transforms) > 4 else ''
        return f"<JX{s} [{labels}{tail}]>"


# ── Factory ──────────────────────────────────────────────────
def jx(data=None, mode='single', name=None):
    t = JXTensor(data=data, name=name)
    if mode == 'vmap':  return t.vmap()
    if mode == 'pmap':  return t.pmap()
    if mode == 'shard': return t.pmap(devices=jax.device_count())
    return t

$ = jx   # jQuery alias


# ── Layers ───────────────────────────────────────────────────
class Linear:
    def __init__(self, in_features, out_features, use_bias=True, key=None):
        key = key if key is not None else jax.random.PRNGKey(0)
        scale = jnp.sqrt(2.0 / (in_features + out_features))
        self.W = jax.random.normal(key, (in_features, out_features)) * scale
        self.b = jnp.zeros(out_features) if use_bias else None
    def __call__(self, x):
        r = x @ self.W
        return r + self.b if self.b is not None else r
    def parameters(self):
        p = [jx(self.W, name='W')]
        if self.b is not None: p.append(jx(self.b, name='b'))
        return p

class Sequential:
    def __init__(self, *layers): self.layers = layers
    def __call__(self, x):
        for l in self.layers: x = l(x)
        return x
    def parameters(self):
        return [p for l in self.layers if hasattr(l, 'parameters') for p in l.parameters()]


# ── Optimizers ───────────────────────────────────────────────
class Optimizer:
    def __init__(self, params, optimizer):
        if params.data is None: raise ValueError("Params need data.")
        self.params    = params
        self.opt       = optimizer
        self.opt_state = optimizer.init(params.data)

    def step(self, loss_fn):
        val, grads = jax.value_and_grad(loss_fn)(self.params.data)
        updates, new_state = self.opt.update(grads, self.opt_state, self.params.data)
        new_params = optax.apply_updates(self.params.data, updates)
        new_opt = Optimizer.__new__(Optimizer)
        new_opt.params    = JXTensor(data=new_params)
        new_opt.opt       = self.opt
        new_opt.opt_state = new_state
        return float(val), new_opt

    def train(self, loss_fn, steps=100, callback=None):
        losses, cur = [], self
        for i in range(steps):
            loss, cur = cur.step(loss_fn)
            losses.append(loss)
            if callback: callback(i, loss, cur.params)
        self.params    = cur.params
        self.opt_state = cur.opt_state
        return losses

def sgd(params, lr=0.01):    return Optimizer(params, optax.sgd(lr))
def adam(params, lr=0.001):  return Optimizer(params, optax.adam(lr))
def adamw(params, lr=0.001): return Optimizer(params, optax.adamw(lr))


# ── Losses ───────────────────────────────────────────────────
def mse(pred, target):    return ((pred - target) ** 2).mean()
def l1_loss(pred, target):return (pred - target).abs().mean()
def cross_entropy(logits, labels): return -(labels * logits.log_softmax()).sum()
def binary_cross_entropy(logits, labels):
    return -(labels * logits.log_sigmoid() + (1 - labels) * (-logits).log_sigmoid()).mean()

print("✓ JX library loaded successfully")

---
## 2. Core Concepts — Lazy Pipelines

A `JXTensor` is a **recipe**, not a result. Operations accumulate as transforms in a list.
Nothing runs until you call an execution method.

In [ ]:
x = jnp.array([1.0, -2.0, 3.0, -4.0, 5.0])

# Build a pipeline — NO computation yet
pipeline = $(x).normalize().relu().softmax()
print("Pipeline object:", pipeline)
print("Transforms queued:", pipeline.inspect()['transforms'])

In [ ]:
# The same pipeline can be reused and extended
extended = pipeline.log().sum()
print("Extended pipeline:", extended)
print("All transforms:", extended.inspect()['transforms'])

---
## 3. Execution Strategies

In [ ]:
# ── value_of() : eager execution ─────────────────────────────
result = $(x).normalize().relu().sum().value_of()
print("value_of():", result)

In [ ]:
# ── jit() : XLA compilation ───────────────────────────────────
# With data attached → compiles AND runs immediately
jit_result = $(x).normalize().relu().sum().jit()
print("jit():", jit_result)

# Without data → returns a compiled JXTensor for repeated use
compiled_pipeline = $(None).normalize().relu().sum().jit()
print("Compiled (no data):", compiled_pipeline)

In [ ]:
# ── vmap() : automatic batching ──────────────────────────────
batch = jnp.stack([x, x * 2, x * 3])   # shape (3, 5)
print("Batch shape:", batch.shape)

batched_result = $(batch).vmap().normalize().relu().sum().value_of()
print("vmap result (one value per sample):", batched_result)

---
## 4. Autograd

In [ ]:
# ── grad() ───────────────────────────────────────────────────
# Differentiates the entire pipeline; always returns a JXTensor
grads = $(x).normalize().relu().sum().grad().value_of()
print("Input :", x)
print("Grads :", grads)

In [ ]:
# ── value_and_grad() : single forward+backward pass ──────────
val_t, grad_t = $(x).normalize().relu().sum().value_and_grad()
print("Value:", val_t.value_of())
print("Grad :", grad_t.value_of())

In [ ]:
# ── hessian() ────────────────────────────────────────────────
x_small = jnp.array([1.0, 2.0, 3.0])
H = $(x_small).sum().hessian().value_of()
print("Hessian shape:", H.shape)
print("Hessian:\n", H)

In [ ]:
# ── jacrev() : full Jacobian ─────────────────────────────────
# f(x) = softmax(x), Jacobian is (n x n)
x_sm = jnp.array([1.0, 2.0, 3.0])
J = $(x_sm).softmax().jacrev().value_of()
print("Jacobian of softmax:", J)

---
## 5. Tensor Operations

In [ ]:
x2d = jnp.arange(12.0).reshape(3, 4)
print("Original (3,4):\n", x2d)

print("\nreshape (2,6):\n",     $(x2d).reshape(2, 6).value_of())
print("\ntranspose (4,3):\n",  $(x2d).transpose().value_of())
print("\nmean axis=1:\n",      $(x2d).mean(axis=1).value_of())
print("\nmax  axis=0:\n",      $(x2d).max(axis=0).value_of())
print("\nargmax axis=1:\n",    $(x2d).argmax(axis=1).value_of())

In [ ]:
v = jnp.array([-2.0, -1.0, 0.0, 1.0, 2.0])

activations = {
    'relu':       $(v).relu().value_of(),
    'sigmoid':    $(v).sigmoid().value_of(),
    'tanh':       $(v).tanh().value_of(),
    'gelu':       $(v).gelu().value_of(),
    'softmax':    $(v).softmax().value_of(),
}

for name, result in activations.items():
    print(f"{name:12s}: {result}")

---
## 6. Binary Operations & Operator Overloading

In [ ]:
a = $(jnp.array([1.0, 2.0, 3.0]))
b = $(jnp.array([4.0, 5.0, 6.0]))

print("a + b  :", (a + b).value_of())
print("a - b  :", (a - b).value_of())
print("a * b  :", (a * b).value_of())
print("a / b  :", (a / b).value_of())
print("a ** 2 :", (a ** 2).value_of())
print("-a     :", (-a).value_of())
print("a * 3  :", (a * 3).value_of())
print("10 - a :", (10 - a).value_of())

In [ ]:
# Matrix multiplication
A = $(jnp.ones((3, 4)))
B = jnp.ones((4, 2)) * 2

result = (A @ B).value_of()
print("A @ B shape:", result.shape)
print(result)

In [ ]:
# Einstein summation
mat = jnp.arange(6.0).reshape(2, 3)
vec = jnp.array([1.0, 2.0, 3.0])

# matrix-vector product via einsum
result = $(mat).einsum('ij,j->i', vec).value_of()
print("einsum 'ij,j->i':", result)

---
## 7. Neural Network Layers

In [ ]:
key = jax.random.PRNGKey(42)

# Single linear layer
layer = Linear(8, 4, key=key)
x_in  = $(jax.random.normal(key, (8,)))

out = layer(x_in)
print("Input JXTensor :", x_in)
print("Output JXTensor:", out)
print("Output value   :", out.value_of())

In [ ]:
# Sequential MLP
key1, key2, key3 = jax.random.split(key, 3)

mlp = Sequential(
    Linear(16, 32, key=key1),
    lambda x: x.relu(),
    Linear(32, 16, key=key2),
    lambda x: x.relu(),
    Linear(16, 1,  key=key3),
)

x_batch = $(jax.random.normal(key, (16,)))
output  = mlp(x_batch)
print("MLP output:", output.value_of())
print("Parameter tensors:", len(mlp.parameters()))

---
## 8. Loss Functions

In [ ]:
pred   = $(jnp.array([2.5, 0.0, 2.0, 8.0]))
target = $(jnp.array([3.0, -0.5, 2.0, 7.0]))

print("MSE :", mse(pred, target).value_of())
print("L1  :", l1_loss(pred, target).value_of())

# Cross entropy (logits + one-hot labels)
logits = $(jnp.array([2.0, 1.0, 0.5, 0.1]))
labels = $(jnp.array([1.0, 0.0, 0.0, 0.0]))
print("CE  :", cross_entropy(logits, labels).value_of())

# Binary cross entropy
b_logits = $(jnp.array([ 1.5, -1.0,  0.5]))
b_labels = $(jnp.array([ 1.0,  0.0,  1.0]))
print("BCE :", binary_cross_entropy(b_logits, b_labels).value_of())

---
## 9. Optimizers & Training Loops

In [ ]:
# Single optimization step
params = $(jnp.array([5.0, -3.0, 2.0]))
target_val = jnp.zeros(3)

def loss_fn(p):
    return jnp.mean((p - target_val) ** 2)

opt = adam(params, lr=0.1)
loss, opt = opt.step(loss_fn)
print(f"After 1 step  — loss: {loss:.4f}  params: {opt.params.data}")

In [ ]:
import matplotlib.pyplot as plt

params = $(jnp.array([5.0, -3.0, 2.0]))

opt    = adam(params, lr=0.1)
losses = opt.train(loss_fn, steps=100)

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title("Training Loss (Adam, 100 steps)")
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final params: {opt.params.data}")
print(f"Final loss  : {losses[-1]:.6f}")

---
## 10. Debugging & Graph Inspection

In [ ]:
# debug_print() is a passthrough — prints stats mid-pipeline
result = (
    $(x)
    .debug_print("raw input")
    .normalize()
    .debug_print("after normalize")
    .relu()
    .debug_print("after relu")
    .sum()
    .value_of()
)
print("Final result:", result)

In [ ]:
# inspect() reveals the full computation graph
pipeline = $(x).normalize().relu().softmax().log().sum()
info = pipeline.inspect()

print("Graph inspection:")
for k, v in info.items():
    print(f"  {k:16s}: {v}")

---
## 11. End-to-End Example — Linear Regression

In [ ]:
import matplotlib.pyplot as plt

# ── Generate synthetic data ───────────────────────────────────
key = jax.random.PRNGKey(0)
N   = 100

X_data = jax.random.uniform(key, (N,), minval=-3, maxval=3)
y_data = 2.5 * X_data - 1.0 + jax.random.normal(key, (N,)) * 0.5

print(f"X shape: {X_data.shape}  y shape: {y_data.shape}")

# ── Model: y = w*x + b ────────────────────────────────────────
# Pack params as [w, b]
theta = $(jnp.array([0.0, 0.0]))

def predict(params, X):
    return params[0] * X + params[1]

def loss_fn(params):
    preds = predict(params, X_data)
    return jnp.mean((preds - y_data) ** 2)

# ── Train ─────────────────────────────────────────────────────
opt    = adam(theta, lr=0.05)
losses = opt.train(loss_fn, steps=300)

w, b = opt.params.data
print(f"\nLearned: y = {w:.3f}x + {b:.3f}")
print(f"True   : y = 2.500x - 1.000")
print(f"Final loss: {losses[-1]:.4f}")

# ── Plot ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(X_data, y_data, alpha=0.4, label='data')
x_line = jnp.linspace(-3, 3, 100)
ax1.plot(x_line, w * x_line + b, 'r-', lw=2, label=f'fit: {w:.2f}x+{b:.2f}')
ax1.set_title('Linear Regression via JX')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(losses)
ax2.set_title('Loss Curve')
ax2.set_xlabel('Step')
ax2.set_ylabel('MSE')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 12. End-to-End Example — MLP on Synthetic Classification Data

In [ ]:
import matplotlib.pyplot as plt

# ── Synthetic 2-class dataset (XOR-like) ─────────────────────
key = jax.random.PRNGKey(7)
k1, k2 = jax.random.split(key)

N = 200
X_cls = jax.random.normal(k1, (N, 2))
y_cls = ((X_cls[:, 0] * X_cls[:, 1]) > 0).astype(jnp.float32)  # XOR boundary

print(f"Dataset: {N} samples, 2 features, binary labels")
print(f"Class balance: {y_cls.mean():.2f}")

# ── MLP with flat parameter vector ───────────────────────────
# For simplicity we use a single Linear layer + sigmoid
# (full MLP with nested params is an exercise for Flax/Equinox)

in_dim, hid_dim, out_dim = 2, 8, 1

# Pack all params: [W1 (2x8 flat), b1 (8), W2 (8x1 flat), b2 (1)]
def init_params(key):
    k1, k2, k3, k4 = jax.random.split(key, 4)
    return jnp.concatenate([
        jax.random.normal(k1, (in_dim * hid_dim,)) * 0.1,
        jnp.zeros(hid_dim),
        jax.random.normal(k3, (hid_dim * out_dim,)) * 0.1,
        jnp.zeros(out_dim),
    ])

def forward(params, X):
    i = 0
    W1 = params[i : i + in_dim * hid_dim].reshape(in_dim, hid_dim); i += in_dim * hid_dim
    b1 = params[i : i + hid_dim];                                    i += hid_dim
    W2 = params[i : i + hid_dim * out_dim].reshape(hid_dim, out_dim);i += hid_dim * out_dim
    b2 = params[i : i + out_dim]
    h  = jax.nn.relu(X @ W1 + b1)
    return jax.nn.sigmoid(h @ W2 + b2).squeeze(-1)

def loss_fn(params):
    preds   = forward(params, X_cls)
    eps     = 1e-7
    return -jnp.mean(y_cls * jnp.log(preds + eps) + (1 - y_cls) * jnp.log(1 - preds + eps))

# ── Train ─────────────────────────────────────────────────────
init_p = init_params(key)
opt    = adam($(init_p), lr=0.01)
losses = opt.train(loss_fn, steps=500)

# ── Accuracy ──────────────────────────────────────────────────
final_preds = forward(opt.params.data, X_cls)
accuracy    = jnp.mean((final_preds > 0.5) == y_cls)
print(f"\nFinal loss    : {losses[-1]:.4f}")
print(f"Train accuracy: {accuracy * 100:.1f}%")

# ── Plot ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Decision boundary
res = 100
xs  = jnp.linspace(-3, 3, res)
ys  = jnp.linspace(-3, 3, res)
XX, YY = jnp.meshgrid(xs, ys)
grid   = jnp.stack([XX.ravel(), YY.ravel()], axis=1)
ZZ     = forward(opt.params.data, grid).reshape(res, res)

ax1.contourf(XX, YY, ZZ, levels=20, cmap='RdBu', alpha=0.6)
ax1.scatter(X_cls[:, 0], X_cls[:, 1], c=y_cls, cmap='RdBu', edgecolors='k', s=20)
ax1.set_title('MLP Decision Boundary')
ax1.grid(True, alpha=0.2)

ax2.plot(losses)
ax2.set_title('Training Loss')
ax2.set_xlabel('Step')
ax2.set_ylabel('BCE Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Summary

| Feature | Method | Returns |
|---|---|---|
| Eager execution | `.value_of()` | JAX array |
| XLA compilation | `.jit()` | array or JXTensor |
| Batching | `.vmap()` | JXTensor |
| Multi-device | `.pmap()` | JXTensor |
| Gradient | `.grad()` | JXTensor |
| Value + gradient | `.value_and_grad()` | (JXTensor, JXTensor) |
| Hessian | `.hessian()` | JXTensor |
| Jacobian (rev) | `.jacrev()` | JXTensor |
| Graph info | `.inspect()` | dict |
| Mid-pipe debug | `.debug_print()` | JXTensor (passthrough) |

---
*JX — Author: J. Roberto Jiménez · tijuanapaint@gmail.com*